# Shoot Window — Phase 1 lab

**Start here in the notebook.** Practice the forecast API in cells. Do not treat this file as the finished app.

## What this notebook is for

The **Shoot Window** app will later be a Streamlit page (`app.py`). First you need to:

1. Send a request to the OpenWeatherMap **forecast** API (not current weather)
2. Receive JSON (a nested dictionary)
3. Extract the fields a photographer cares about: temperature, wind, clouds, rain chance, sunset

An **API** is a way for your program to ask another service for data.  
An **endpoint** is the specific URL you call.  
**JSON** is the nested data format that comes back.

The finished product is **not** "here is the weather". It is:  
`Weather Data → Analysis → Photography Score → Recommendation → Decision`

After the cells below work, copy the ideas into `photo_planner/` (Person A: client + models + scoring; Person B: charts + `app.py`).

Docs: [5 day / 3 hour forecast](https://openweathermap.org/forecast5)

## 0. API key (do not commit this)

An **API key** is a secret password OpenWeatherMap gives you. Your code sends it with every request. Never paste it into a cell you will commit to GitHub.

Create a free key: https://home.openweathermap.org/api_keys

- **Colab:** store it in Secrets as `OPENWEATHER_API_KEY`, then read with `userdata`.
- **Local Jupyter:** put it in `../.env` and load with `python-dotenv`.

The next cell is infrastructure (`# KEEP`). Your assignment logic starts at section 1.

In [1]:
# KEEP — this cell only loads the key. Do not print API_KEY.
import os
import getpass

# Option A — type it once in this session (it will live in the notebook output if you print it: don't).
API_KEY = os.environ.get("OPENWEATHER_API_KEY") or getpass.getpass("OPENWEATHER_API_KEY: ")

assert API_KEY, "Set the API key before calling the API"

## 1. Connect to the forecast API

**Your only goal in the next cell:** given a city name, receive a response and print `status_code`.

Endpoint (the URL you call):
`https://api.openweathermap.org/data/2.5/forecast`

This is the **5-day / 3-hour** forecast. Photographers need several hours so we can recommend a shooting window. Do **not** use `/data/2.5/weather` (that is only "now").

Query params (the extra pieces attached to the URL):

- `q` — city name (try `Tel Aviv,IL`)
- `appid` — your API key
- `units=metric` — temperatures in Celsius

HINT: `requests.get(url, params={...})` sends a GET request. Print `status_code` first (200 means OK), not the whole JSON.

In [2]:
import requests  # KEEP — library that sends HTTP requests

FORECAST_URL = "https://api.openweathermap.org/data/2.5/forecast"  # KEEP
CITY = "Tel Aviv"  # KEEP — you can change this later

# YOUR CODE GOES HERE 👇
# Goal: send GET with params q, appid, units and print status_code.
params = {
    "q": CITY,
    "appid": API_KEY,
    "units": "metric"
}

response = requests.get(FORECAST_URL, params=params)

print(response.status_code)
# HINT: params is a dict, for example {"q": CITY, "appid": API_KEY, "units": "metric"}
#
# TODO (STUDENT): Write the requests.get call and print response.status_code.
# response = ...
# print(response.status_code)

200


## 2. Request a forecast for a city

Wrap the request in a function so you can reuse it.

`city` is one string, for example `"Tel Aviv"`.  
The function should return the JSON as a Python `dict`.

Try a real city and a fake one. What status code is **404** vs **401**?

When this works, the same idea will live in `photo_planner/client.py` (`OpenWeatherClient.get_forecast`).

In [4]:
def get_forecast(city: str) -> dict:
    """
    Receive one city name.
    Contact the forecast API.
    Return the JSON dict.
    """

    params = {
        "q": city,
        "appid": API_KEY,
        "units": "metric"
    }

    response = requests.get(FORECAST_URL, params=params)

    if response.status_code != 200:
        raise Exception(
            f"Request failed with status code {response.status_code}"
        )

    return response.json()


payload = get_forecast("Tel Aviv")
print(payload["city"]["name"], len(payload["list"]))

Tel Aviv 40


## 3. Process JSON — photographer fields, not a weather dump

JSON is nested. You walk keys like a dictionary.

The forecast has two important parts:

- `payload["city"]` — name, country, sunrise, sunset
- `payload["list"]` — about 40 slots, one every 3 hours

| Need | Typical path |
| --- | --- |
| city | `city["name"]` |
| sunset | `city["sunset"]` (unix time) |
| slot time | `list[i]["dt_txt"]` |
| temperature | `list[i]["main"]["temp"]` |
| wind | `list[i]["wind"]["speed"]` |
| cloud cover | `list[i]["clouds"]["all"]` (0–100) |
| rain chance | `list[i]["pop"]` (0–1, so 0.4 means 40%) |
| rain mm | `list[i]["rain"]["3h"]` — this key is often **missing** |

You can also practice on `../data/sample_forecast.json` without the internet.

When this cell is clean, copy the mapping into `photo_planner/models.py`.

In [5]:
city = payload["city"]["name"]
sunset = payload["city"]["sunset"]

first_slot = payload["list"][0]

time = first_slot["dt_txt"]
temperature = first_slot["main"]["temp"]
wind = first_slot["wind"]["speed"]
clouds = first_slot["clouds"]["all"]
rain_probability = first_slot["pop"]

print("City:", city)
print("Sunset:", sunset)
print("Time:", time)
print("Temperature:", temperature, "°C")
print("Wind:", wind, "m/s")
print("Cloud cover:", clouds, "%")
print("Rain probability:", rain_probability * 100, "%")

City: Tel Aviv
Sunset: 1788710390
Time: 2026-09-06 15:00:00
Temperature: 30.18 °C
Wind: 4.1 m/s
Cloud cover: 0 %
Rain probability: 0 %


## 4. Look at several hours (the reason we use forecast)

A photographer booked 17:00. The free API gives 12:00, 15:00, 18:00, 21:00...

Print a small table for tomorrow's slots: time, temperature, wind, clouds, pop.

Do **not** invent a Photography Score here yet. First see the raw numbers. Tomorrow, in `scoring.py`, you will turn these numbers into 0–100.

HINT: loop `for slot in payload["list"]:` and keep slots whose `dt_txt` starts with the date you care about.

In [7]:
# Choose one date from the forecast
target_date = payload["list"][0]["dt_txt"][:10]

for slot in payload["list"]:
    if slot["dt_txt"].startswith(target_date):
        time = slot["dt_txt"]
        temp = slot["main"]["temp"]
        wind = slot["wind"]["speed"]
        clouds = slot["clouds"]["all"]
        pop = slot["pop"] * 100

        print(
            f"{time} | "
            f"Temp: {temp}°C | "
            f"Wind: {wind} m/s | "
            f"Clouds: {clouds}% | "
            f"Rain: {pop:.0f}%"
        )

2026-09-06 15:00:00 | Temp: 30.18°C | Wind: 4.1 m/s | Clouds: 0% | Rain: 0%
2026-09-06 18:00:00 | Temp: 29.5°C | Wind: 1.83 m/s | Clouds: 0% | Rain: 0%
2026-09-06 21:00:00 | Temp: 27.93°C | Wind: 1.88 m/s | Clouds: 0% | Rain: 0%


## 5. First visualization (optional in the notebook)

Do this **after** you can print temperatures for several hours. Person B will later move charts into `photo_planner/charts.py`.

In the notebook, Matplotlib or Plotly are both fine. In the final app, prefer Plotly (`st.plotly_chart`).

Idea: a simple line of temperature (or cloud cover) across the day. The Photography Score chart comes later, after `scoring.py` exists.

In [ ]:
# YOUR CODE GOES HERE 👇
# OPTIONAL in the notebook — required later in charts.py / app.py
#
# TODO (STUDENT): plot temperature across the slots you printed above

## 6. Next (outside this notebook)

Stop treating this notebook as the product once the functions exist in the package.

1. Person A: `photo_planner/models.py` + tests on `data/sample_forecast.json`
2. Person A: `client.py` + `validation.py`
3. Both: agree Photography Score weights on paper
4. Person A: `scoring.py`
5. Person B: `charts.py` + `app.py`
6. Together: GitHub + Streamlit Community Cloud (`docs/DEPLOY.md`)

**One next task after this notebook:** do not open Streamlit yet. Finish a working `requests.get` here, then Person A implements `from_slot_json` using the sample JSON.